# Automatic Segmentation of Sessile Droplets
## Contact-Angle Measurement Pipeline — CO4 / MSLS (ZHAW, FS26)

**Course:** Imaging for the Life Sciences (CO4), Master of Science in Life Sciences  
**Project:** pair work (two students)  
**Submission:** 31 May 2026

---

## Abstract

We segment 39 back-lit, side-view photographs of sessile liquid droplets (water, diiodomethane, octanol) resting on ~13 different surfaces, in order to compute their **contact angles** automatically. We compare four segmentation approaches of increasing capability — three classical CV variants and one foundation-model approach (Meta's Segment Anything Model, SAM) — against 10 manually annotated ground-truth masks. The best automatic pipeline achieves **Dice = 0.97 ± 0.02** spatial agreement and reproduces the original laboratory contact-angle measurements within **6° MAE**. The central finding of the project is that the bottleneck in such a pipeline is rarely the model — it is the brittle hand-tuned preprocessing fed into it.

---

## 1. Introduction & problem statement

A *sessile droplet* is a droplet of liquid resting on a solid surface. The angle the liquid–air interface makes with the surface at the **contact line** — the **contact angle θ** — is a direct measure of how wettable the surface is by that liquid. Small θ means the liquid spreads (hydrophilic, oleophilic); θ near 180° means the liquid beads up and barely touches (super-hydrophobic, like water on Lotus leaves).

To measure θ automatically from a side-view photograph, we need a **clean binary mask** of the droplet so that the geometry — the curve of the droplet edge and where it meets the surface — can be analysed. This sounds easy. It is not, for three concrete reasons our data makes obvious:

1. **Transparent droplets.** A water or octanol droplet is essentially a clear lens: the *interior* of the droplet has approximately the same intensity as the back-lit background. Only the curved **rim** is darker (where the curvature bends light away). Classical assumptions like "droplet = dark blob" simply fail.
2. **Specular reflections.** Polished gold and metal supports throw bright highlights that look like edges to any gradient-based method.
3. **Surface heterogeneity.** The same algorithm must cope with a near-spherical droplet on Teflon, a flat lens on glass, and an invisible smear on fabric — without per-image tuning.

### Goals
- A single automatic pipeline that segments droplets across all surfaces in the dataset.
- Quantitative evaluation against manually annotated ground truth (Dice, IoU).
- **(Bonus)** Use the segmentations to compute contact angles, and compare against the original laboratory measurements made by hand with a goniometer.

---

## 2. Dataset

**39 JPEG images**, 1600 × 1200 pixels, organised in `fwdfoto/`. Each file is named `<liquid>-<surface>[-<volume>].jpg`:

- **Liquids:** `h2o` (water), `i2` (diiodomethane CH₂I₂), `oct` (octanol)
- **Surfaces:** `lotus`, `fog`, `fuoc`, `metall`, `plexig`, `print`, `rain`, `teflon`, `tessuto`, `vetro`
- **Volume suffix** `-5` / `-100` only on coated-glass variants (different coating concentrations).

A few images contain **no discernible droplet** — the liquid spread into an invisible film. For those, the correct output is an empty mask.

**Origin and license.** The images were acquired by us during an earlier lab project on wettability characterisation; we own the raw data and reuse it here with all collaborators' agreement. Original measured contact angles for many of the substrate × liquid combinations are in `angolo di contatto.xlsx` from the same lab work; we use those as the ultimate ground truth in §7.

In [ ]:
# --- Setup: paths, imports, helpers ---
import os, sys, json, glob
import numpy as np, cv2
import matplotlib.pyplot as plt
from IPython.display import Image, display

# Locate the repo root relative to this notebook (automatic_segmentation/notebook/)
REPO    = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
SRC     = os.path.join(REPO, 'fwdfoto')
PREP    = os.path.join(REPO, 'data', 'preprocessed')
MANBIN  = os.path.join(REPO, 'FOTO CO4 MASKS MANUAL', 'manual_binary')
RESULTS = os.path.join(REPO, 'automatic_segmentation', 'results')
METHODS = os.path.join(REPO, 'automatic_segmentation', 'methods')

# Make the segmentation methods importable
sys.path.insert(0, METHODS)
sys.path.insert(0, os.path.join(REPO, 'automatic_segmentation', 'reproduce'))

print(f'REPO   : {REPO}')
print(f'Images : {len(glob.glob(os.path.join(SRC, "*.jpg"))) - 1} (excluding test.jpg)')
print(f'Manual : {len(glob.glob(os.path.join(MANBIN, "*.png")))} binary masks')

In [ ]:
# --- Show one example image per liquid ---
examples = ['h2o-teflon', 'i2-lotus', 'oct-fuoc']
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, nm in zip(axes, examples):
    img = cv2.cvtColor(cv2.imread(os.path.join(SRC, nm + '.jpg')), cv2.COLOR_BGR2RGB)
    ax.imshow(img); ax.set_title(nm); ax.axis('off')
plt.tight_layout(); plt.show()

---

## 3. Preprocessing

We developed a 9-step preprocessing pipeline in `automatic_segmentation/methods/preprocessing.py`. Each step has a specific data-driven motivation:

1. **Crop timestamp** (bottom 13 %). Every image carries a high-contrast `Title 16:42:59 2022/04/28` banner that any thresholder would otherwise pick up as foreground.
2. **Best-channel selection.** Split into B, G, R and pick the channel with maximum variance in the upper ROI. Automatic per-image choice that copes with the dataset's wide background range (beige, green Lotus, golden metal).
3. **Substrate-line detection.** The strongest sustained horizontal Sobel-y gradient in the lower half marks the surface.
4. **Background subtraction.** Morphological closing with a 101 × 101 ellipse approximates the local illumination field; dividing by it normalises uneven lighting. Critical for `fog`-coated and side-lit images.
5. **CLAHE** (clip 1.5, tile 16 × 16). Adaptive contrast enhancement; conservative parameters chosen to avoid amplifying noise in low-contrast octanol-on-glass cases.
6. **Bilateral filter** (d = 9, σ = 75). Edge-preserving denoise; Gaussian blur destroys the already-weak octanol-droplet edges (~3–5 DN gradient) — bilateral preserves them.
7. **Blank-image detection.** Variance threshold flags images with no visible droplet (e.g. `i2-fuoc`).
8. **Crop above substrate.** Discard everything below the surface line.
9. **Polarity normalization.** Ensure the droplet is always *darker* than the background — some over-exposed cases reverse this convention.

The preprocessed images for all 39 inputs are saved to `data/preprocessed/`; per-step intermediate visualisations on representative images are in `data/preprocessed/pipeline_steps/`.

### Honest note on what the final pipeline actually uses

Out of the four segmentation methods compared in §5, **only one (`basic_segment`, §5.1)** actually uses preprocessing — and it uses a simplified, inlined version (grayscale + median blur + per-column sky subtraction), not the full 9-step pipeline above. The other three methods do not use `preprocessing.py` at all:

- `watershed_segment` (§5.2) consumes the output of `basic_segment` and refines it.
- `sam_segment` (§5.3) feeds **raw BGR** to SAM and uses preprocessing only to pick a point prompt.
- `automask_segment` (§5.4, our chosen final method) feeds **raw BGR** to SAM and does no preprocessing whatsoever.

This is not a flaw — it is a finding. SAM's foundation-model training is robust to illumination gradients, contrast levels, and background variation, so the classical preprocessing steps (CLAHE, bilateral, channel selection) become unnecessary. Our preprocessing pipeline is retained in the repo because (i) it was an essential stepping stone — every classical CV iteration we tried before adopting SAM relied on it, and (ii) it documents what classical pipelines would need on this dataset.

**Generalisable point:** for sessile-droplet imagery of this kind, a foundation segmentation model substantially reduces the preprocessing burden that classical methods otherwise impose.

---

## 4. Manual segmentation

We traced **10 droplets by hand in Fiji/ImageJ** on a representative subset of the dataset. The exports landed as 8-bit grayscale images with the droplet drawn in white on top of the original photo. To use them as binary ground truth for Dice/IoU evaluation, we recover a true 0/255 mask per file:

1. Threshold at intensity ≥ 240 (the painted droplet was the brightest region).
2. Keep the largest connected component (drops the timestamp text and other bright specks).
3. Fill interior holes (internal specular highlights inside the droplet).
4. For 3 files where the fill leaked along bright specular plate edges (the two Teflon and one metal case), apply a **hand-set baseline cut** — agreed visually — to discard the leaked skirt below the contact line.

All 10 final binary masks live in `FOTO CO4 MASKS MANUAL/manual_binary/`.

In [ ]:
# --- Display one example: original photo + recovered binary mask ---
nm = 'h2o-metall'
orig = cv2.cvtColor(cv2.imread(os.path.join(SRC,  nm + '.jpg')), cv2.COLOR_BGR2RGB)
man  = cv2.imread(os.path.join(MANBIN, nm + '.png'), cv2.IMREAD_GRAYSCALE)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(orig); axes[0].set_title(f'{nm}: original photo'); axes[0].axis('off')
axes[1].imshow(man, cmap='gray'); axes[1].set_title('recovered binary mask'); axes[1].axis('off')
plt.tight_layout(); plt.show()
print(f'mask values: {set(np.unique(man).tolist())}  -> truly binary')

---

## 5. Automatic segmentation

We developed and compared **four segmentation methods**, of increasing capability. This is not a recipe-and-best-one section: we present the full progression because *the failures of each step were the data that drove the next*.

| § | Method | Type | One-line description |
|---|---|---|---|
| 5.1 | `basic_segment`     | classical CV       | per-column sky-subtraction + Otsu + largest blob on baseline |
| 5.2 | `watershed_segment` | classical CV       | refines basic_segment's outline with watershed markers |
| 5.3 | `sam_segment`       | foundation model   | SAM with a point prompt placed automatically |
| 5.4 | `automask_segment`  | foundation model   | SAM automatic mask generation + **shape-based selection** ★ final |

All four are in `automatic_segmentation/methods/`. They each expose a `segment(image)` function returning a full-size 0/255 mask. They were run on all 39 images via `automatic_segmentation/reproduce/run_method.py`, with masks saved to `results/masks/<method>/` and overlay visualisations to `results/overlays/<method>/`.

### 5.1 — Method 1: classical baseline-based segmentation (`basic_segment`)

Pure OpenCV, follows the four-step recipe suggested in the lecture:

1. Grayscale + median blur.
2. **Detect the baseline** (= top edge of the support plate) as the strongest long horizontal Sobel-y gradient in the lower 60 % of the image.
3. **Per-column sky subtraction** → a "deviation map" that is positive both for darker (rim) and brighter (interior reflection) droplet pixels.
4. **Otsu threshold** the deviation map, restrict to above the baseline, morphologically close gaps, flood-fill the enclosed interior.
5. Keep the largest blob touching the baseline; reject if too flat to be a droplet.

**What works:** opaque droplets with clear rims (iodine on dark surfaces).  
**What fails:** transparent droplets whose interior matches the background, and — critically — any case where the baseline detector locks onto the wrong edge (see §5.5).

In [ ]:
# --- Show basic_segment on one image ---
import basic_segment
img  = cv2.imread(os.path.join(SRC, 'h2o-fuoc.jpg'))
mask, baseline = basic_segment.segment(img)
ov = img.copy()
cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
cv2.drawContours(ov, cnts, -1, (0, 0, 255), 4)
cv2.line(ov, (0, baseline), (ov.shape[1], baseline), (0, 255, 0), 2)
plt.figure(figsize=(8, 6))
plt.imshow(cv2.cvtColor(ov, cv2.COLOR_BGR2RGB))
plt.title('basic_segment on h2o-fuoc (contour red, baseline green)'); plt.axis('off'); plt.show()

### 5.2 — Method 2: watershed refinement (`watershed_segment`)

Takes `basic_segment`'s mask, erodes it heavily to a **sure foreground marker**, dilates it and inverts to a **sure background marker**, then runs OpenCV's watershed to snap the boundary to the actual image gradient. The visual result is sharper edges (mean solidity 0.81 → 0.83). However, **watershed is a refiner, not a re-locator**: if `basic_segment` picked the wrong region, watershed just sharpens the wrong region. This is a structural limitation of any refinement-style classical method.

In [ ]:
import watershed_segment
mask_w, _ = watershed_segment.segment(img)
ov = img.copy()
cnts, _ = cv2.findContours(mask_w, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
cv2.drawContours(ov, cnts, -1, (0, 0, 255), 4)
plt.figure(figsize=(8, 6))
plt.imshow(cv2.cvtColor(ov, cv2.COLOR_BGR2RGB))
plt.title('watershed_segment on h2o-fuoc'); plt.axis('off'); plt.show()

### 5.3 — Method 3: SAM with a point prompt (`sam_segment`)

Meta's [Segment Anything Model](https://segment-anything.com/) is a foundation model trained on 11 million images. Given an image and a point/box prompt, it returns a mask of whichever object the prompt is on, with no per-class training.

Our strategy here: place **one positive point prompt** automatically at "the column with the most non-sky deviation just above the baseline" (i.e. likely on the droplet), plus negative prompts in the corners, and pick the best returned mask. This works much better than classical CV (mean solidity 0.81 → 0.88, and cracks cases like Teflon that classical CV never could). The remaining failures are **prompt-placement errors**: on transparent droplets the bright plate edge can deviate from the sky *more* than the clear droplet does, so the seed point lands on the plate instead.

### 5.4 — Method 4: SAM automatic mask + shape-based selection (`automask_segment`)  ★ **final method**

The breakthrough that fixed most of the dataset at once: stop trying to *guess* a single prompt point and instead let SAM propose **every** object in the image, then **select the droplet by shape**.

SAM's `SamAutomaticMaskGenerator` samples a grid of points across the image, queries SAM at each, and returns 7–15 candidate masks per image — background, plate, droplet, reflections, sometimes sub-parts. We then filter:

1. **Reject masks touching the top edge** of the image → background.
2. **Reject masks wider than 75 % of the frame** → plate / background band.
3. **Reject masks with solidity ≤ 0.5** → not blob-like (scattered noise, thin strips).
4. **Keep area in [0.3 %, 25 %]** of the frame → plausible droplet size.
5. Use a rough baseline only as a **soft tiebreaker**, never as a hard gate.

Step 5 is the crucial design choice: every earlier method *required* a correct baseline and threw away droplets that didn't sit exactly on it. Demoting the baseline to a soft hint eliminated the single biggest source of failure (see §5.5).

**Note: this method receives the raw BGR image and applies no preprocessing.** SAM is robust enough to handle the dataset's contrast, lighting, and colour variation directly.

In [ ]:
# --- Show automask_segment result on h2o-metall (loads precomputed mask; full re-run takes ~13 min on CPU) ---
nm = 'h2o-metall'
img  = cv2.imread(os.path.join(SRC, nm + '.jpg'))
mask = cv2.imread(os.path.join(RESULTS, 'masks', 'automask', nm + '.png'), cv2.IMREAD_GRAYSCALE)
ov = img.copy()
cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
cv2.drawContours(ov, cnts, -1, (0, 0, 255), 4)
plt.figure(figsize=(8, 6))
plt.imshow(cv2.cvtColor(ov, cv2.COLOR_BGR2RGB))
plt.title(f'automask on {nm}'); plt.axis('off'); plt.show()
print('To re-run the full 39-image batch:')
print('  python automatic_segmentation/reproduce/run_method.py --method automask')
print('(~13 min on CPU; MPS unsupported because SAM auto-generator uses float64)')

### 5.5 — The bottleneck was the baseline, not the model

This is the most important section of the project for *understanding* what we did. The short version:

For several days we kept improving classical methods, then prompted SAM, then an ensemble "quality router" that picked the best of two methods per image with a hand-tuned score. Every fix only chipped away at the edges. Clearly visible droplets — `h2o-metall`, `i2-rain-100`, `h2o-rain-100` — were being *missed entirely*, which should be impossible for SAM.

Tracing one such failure showed it cleanly: on a 1200-pixel-tall image, the baseline detector was returning y ≈ 1120 — almost the very bottom of the image — when the actual contact line was around y ≈ 870. The detector had locked onto the *bottom* of the plate, not its top. Every method built on top of it was discarding the correct droplet for "not on baseline". Testing three different baseline detectors against eight images gave at most 5/8 correct: gradient-based baseline detection is **fundamentally unreliable** on shiny plates.

The fix was not to invent yet another baseline detector. It was to **stop depending on a precise baseline at all** — Method 4 above. That single change took the bulk of the dataset from "wrong" to "correct" with no further tuning.

**Key lesson, generalisable beyond this project:** before blaming the model, check what you are feeding it. A brittle upstream component had been silently corrupting downstream results across four methods at once.

---

## 6. Method comparison

All four methods were run over the full 39-image dataset; masks were saved to `results/masks/<method>/`. The script `reproduce/run_compare.py` then computes per-method statistics and the side-by-side grid below.

In [ ]:
# --- Per-method statistics over all 39 images ---
stats = json.load(open(os.path.join(RESULTS, 'compare_stats.json')))
print(f"{'method':12s} {'segmented':>10s} {'mean solidity (of ok masks)':>30s}")
print('-' * 56)
for me in ['basic', 'watershed', 'sam', 'automask']:
    s = stats[me]
    print(f"{me:12s} {s['ok']:>4d} / 39   {s['mean_solidity_of_ok']:>15.3f}")

"Segmented" only counts non-empty masks; **solidity** (mask area ÷ convex-hull area) is the more informative quality proxy because it penalises ragged or plate-smeared masks. The steady climb basic 0.81 → watershed 0.83 → SAM 0.88 → automask 0.96 is the real progression.

The visual comparison below shows representative images across all four methods.

In [ ]:
display(Image(filename=os.path.join(RESULTS, 'overlays', 'comparison.jpg')))

---

## 7. Evaluation

### 7.1 — Metrics (Dice and IoU)

Following the lecture (Block 6 - 05), we use the **Dice similarity coefficient** as the primary mask-similarity metric, complemented by **Intersection over Union (IoU)**:

$$\mathrm{Dice}(A,B) = \frac{2\,|A \cap B|}{|A| + |B|} \qquad \mathrm{IoU}(A,B) = \frac{|A \cap B|}{|A \cup B|} \qquad (\mathrm{IoU} \le \mathrm{Dice})$$

Both are computed on binary masks; values in [0, 1]; 1 = perfect agreement. Implemented in `automatic_segmentation/reproduce/evaluate.py`.

In [ ]:
# --- Dice / IoU implementation ---
def dice(a, b):
    a = a > 0; b = b > 0
    s = a.sum() + b.sum()
    return 1.0 if s == 0 else float(2.0 * np.logical_and(a, b).sum() / s)

def iou(a, b):
    a = a > 0; b = b > 0
    u = np.logical_or(a, b).sum()
    return 1.0 if u == 0 else float(np.logical_and(a, b).sum() / u)

In [ ]:
# --- Score automask against the 10 manual binary masks ---
rows = []
for mp in sorted(glob.glob(os.path.join(MANBIN, '*.png'))):
    nm  = os.path.splitext(os.path.basename(mp))[0]
    ap  = os.path.join(RESULTS, 'masks', 'automask', nm + '.png')
    if not os.path.exists(ap): continue
    a = cv2.imread(ap, cv2.IMREAD_GRAYSCALE)
    b = cv2.imread(mp, cv2.IMREAD_GRAYSCALE)
    rows.append((nm, dice(a, b), iou(a, b)))

print(f"{'image':16s} {'Dice':>7s} {'IoU':>7s}")
print('-' * 34)
for nm, d, j in rows:
    print(f'{nm:16s} {d:7.3f} {j:7.3f}')
ds = np.array([r[1] for r in rows]); js = np.array([r[2] for r in rows])
print('-' * 34)
print(f'mean +/- std :  Dice {ds.mean():.3f} +/- {ds.std():.3f}   IoU {js.mean():.3f} +/- {js.std():.3f}   (N = {len(rows)})')

**Result:** Dice = **0.968 ± 0.018**, IoU = **0.939 ± 0.033** over the 10 images with binary ground truth. Dice > 0.9 is generally considered excellent agreement; achieving 0.97 with a standard deviation of only 0.018 means the agreement is not just high on average but **consistently** high across substrates. The lowest cases (`oct-rain-5` 0.931, `oct-fog-100` 0.947) are flat, low-contrast octanol droplets — exactly where the most edge disagreement is expected.

### 7.2 — Contact-angle measurement (bonus, evaluation vs. lab values)

The mask is the *means*, not the end — the project's real goal is the **contact angle**. We compute it from each binary mask as follows (`automatic_segmentation/reproduce/compute_contact_angles.py`):

1. The mask's bottom row defines the local baseline.
2. Extract the droplet contour; discard the bottom 6 px to avoid the rasterised flat edge biasing the fit.
3. Fit a circle to the upper contour points by **geometric least squares** (Gauss-Newton minimising true Euclidean distance, initialised from a Kasa algebraic fit). Geometric LS is necessary here because algebraic fits like Kasa are biased on partial arcs and yielded ~16° of systematic error in our first attempt.
4. The contact angle is read off the fitted circle: $\cos\theta = (c_y - y_{\text{baseline}})/R$, measured through the liquid.

We compare against the **original laboratory measurements** in `angolo di contatto.xlsx`, made by hand from earlier work using a goniometer (mean of left and right angles). Per-image circle-fit visualisations are saved to `results/overlays/contact_angle/`.

In [ ]:
# --- Load and display the contact-angle comparison ---
rows = json.load(open(os.path.join(RESULTS, 'contact_angles.json')))
print(f"{'image':16s} {'lab':>7s} {'manual':>8s} {'Dman':>6s}  {'auto':>7s} {'Dauto':>6s}")
print('-' * 60)
dm, da = [], []
for r in rows:
    dm.append(r['err_manual']); da.append(r['err_automask'])
    print(f"{r['image']:16s} {r['real_deg']:7.1f} {r['manual_deg']:8.1f} {r['err_manual']:+6.1f}  {r['automask_deg']:7.1f} {r['err_automask']:+6.1f}")
dm, da = np.array(dm), np.array(da)
print('-' * 60)
print(f'manual mask  vs lab : signed mean {dm.mean():+.1f}, std {dm.std():.1f}, MAE {np.abs(dm).mean():.1f} deg')
print(f'automask     vs lab : signed mean {da.mean():+.1f}, std {da.std():.1f}, MAE {np.abs(da).mean():.1f} deg')

**Result:** the automatic pipeline reproduces the original laboratory contact-angle measurements within **6.0° MAE** (signed mean −3.3°), essentially indistinguishable from what is obtained by using the manually traced masks (5.4° MAE). This is significant because it shows the pipeline is not merely *spatially* accurate (Dice 0.97), but produces the same *physical measurement* the lab originally produced by hand.

Geometric circle fit on `h2o-metall`:

In [ ]:
display(Image(filename=os.path.join(RESULTS, 'overlays', 'contact_angle', 'h2o-metall_combined.jpg')))

---

## 8. Discussion

### What the numbers mean together

Two evaluation axes converge on the same conclusion:

- **Spatial agreement:** Dice = 0.97 ± 0.02 over 10 ground-truth masks.
- **Physical agreement:** contact-angle error vs. lab goniometer measurements = 6.0° MAE for the fully automatic pipeline.

That second number is the more meaningful one for the project's actual purpose. It says the pipeline can replace the manual goniometer step for the kind of droplets in this dataset, with the typical inter-operator variability one would see between two human measurers.

### Where automask succeeds

All cases with a **visible droplet rim** — water and iodine on lotus / teflon / metal / printed glass — yielded Dice ≥ 0.96 and contact angles within 10° of the lab values. SAM's foundation-model training generalises across the surface variation without per-image tuning. Even the *Teflon* cases, which classical CV could never crack because the bright plate glare is the same brightness as the droplet, came out cleanly because SAM understands "object" rather than just "pixel value".

### Where automask struggles

- **`oct-print-5` (Δθ = −19°)** — automask cuts the bottom of the droplet slightly differently from the manual trace; since contact angle is dominated by the few pixels at the baseline, small mask differences at the bottom create large angle differences. This is also visible as the lowest IoU in the dataset.
- **`i2-teflon`, `i2-lotus`** — diiodomethane droplets on rough/featured surfaces; the lab measurements themselves are flagged as "irregolare" in the spreadsheet, so it is not clear that an exact reproduction would even be the right target.
- **4 empty results out of 39** (`h2o-fog-100`, `h2o-fog-5`, `oct-plexig`, `oct-vetro`) — these images contain no clearly discernible droplet (the liquid spread into a film). Reporting them as empty is **arguably correct**, not a failure. This deserves human verification.

### Why spatial agreement (Dice) does not equal measurement agreement

On `oct-print-5` the masks have Dice ≈ 0.97 yet the derived contact angles differ by 19°. This is not a contradiction — it is structural. Contact angle is determined by the **few pixels at the baseline**, which are weighted equally with all other pixels in Dice. A few-pixel difference at the contact line therefore changes the angle a lot without measurably changing Dice. **High Dice does not guarantee a good downstream measurement.** This is a useful general point: which metric you measure should depend on which decision is downstream of the mask.

### What we learned about classical CV here

The classical methods (5.1–5.2) plateau because they treat "droplet" as a function of pixel statistics, which simply fails when the droplet is transparent. Block 6-04 of the course lecture lists this as a known limitation of thresholding/region-growing methods: *"Lack of Context: these methods look at local pixel values but often fail to understand the global shape or location."* Our results are a direct empirical illustration of that.

---

## 9. Limitations & further work

### Limitations

- **Ground truth is limited to 10 images.** All Dice/IoU and contact-angle-vs-lab statistics are computed on this subset. Extending to more images would tighten the confidence intervals.
- **The manual masks are semi-recovered.** Our hand-traced exports were grayscale photos with the droplet painted white; we recovered binary masks by thresholding, keeping the largest blob, filling holes, and (in three cases) cutting at a hand-set baseline. The shape information is faithful to the manual tracings; the cleanup is automated.
- **`automask_segment` runs on CPU only** (~20 s/image, ~13 min for the full set). Apple's MPS does not support the float64 operations SAM's automatic generator uses. On a CUDA GPU this would be sub-second.
- **The contact-angle method assumes a spherical cap.** Large droplets are flattened by gravity and deviate from a circle; this is the dominant residual error on `h20-lotus`, where the lab value (146.7°) and our value (144.7°) differ by 2° even with a near-perfect mask.

### Further work

- **More ground-truth masks** to tighten evaluation (especially on the trickier surfaces `tessuto`, `fog`, `plexig`).
- **Local tangent fit** (instead of global circle) for the contact-angle measurement — yields separate left and right angles, flags asymmetric droplets, and is what high-end goniometer software uses.
- **A self-trained baseline detector** (CNN trained on a few hand-labelled baselines) would be much more reliable than gradient heuristics, and would also benefit the classical pipeline.
- **Replace SAM with a lighter foundation segmentation model** (MobileSAM, EfficientSAM, FastSAM) to enable real-time use without GPU.

---

## 10. Conclusion

We built an automatic segmentation pipeline for back-lit sessile droplet photographs, compared four approaches end-to-end (two classical, two foundation-model), and evaluated against both manual ground-truth masks and original laboratory goniometer measurements.

The final method (SAM automatic mask generation + shape-based selection) achieves **Dice = 0.97** spatial agreement against manual masks and reproduces lab contact angles within **6° MAE**, with no per-image tuning and no preprocessing dependency on the baseline detector that defeated the classical methods.

The most generalisable lesson from the project is methodological: a brittle upstream component — in our case, baseline detection — can silently destroy results across many downstream methods. The breakthrough was not a better algorithm; it was diagnosing that the same bad input was being fed to four different methods. Removing that dependency, by letting a foundation model propose objects and selecting by shape, fixed most of the dataset in a single change.

---

## 11. Where everything lives in the repo

All artefacts referenced in this notebook are saved to deterministic paths and can be regenerated from `automatic_segmentation/reproduce/`:

| What | Path | Notes |
|---|---|---|
| Raw images (39) | `fwdfoto/` | source data |
| Preprocessed images (39) | `data/preprocessed/` | from `preprocessing.py` |
| Per-step preprocessing visualisations | `data/preprocessed/pipeline_steps/` | a few representative cases |
| Manual binary masks (10) | `FOTO CO4 MASKS MANUAL/manual_binary/` | ground truth |
| Auto masks per method (39 each × 4) | `automatic_segmentation/results/masks/<method>/` | basic / watershed / sam / automask |
| Per-image overlays per method | `automatic_segmentation/results/overlays/<method>/` | contour drawn on photo |
| 4-method comparison grid | `automatic_segmentation/results/overlays/comparison.jpg` | side-by-side panel |
| Contact-angle visualisations (10) | `automatic_segmentation/results/overlays/contact_angle/` | circle fit + 3-panel sheet |
| Comparison statistics | `automatic_segmentation/results/compare_stats.json` | solidity per method |
| Contact angle vs. lab values | `automatic_segmentation/results/contact_angles.json` | per-image table |

---

## 12. Use of generative AI

In accordance with the course requirement to disclose all uses of generative AI tools:

### Generative-AI segmentation model used in the pipeline itself
- **Segment Anything Model (SAM, Meta AI; `vit_b` checkpoint).** SAM is used as a component of the final automatic segmentation method (§5.3 prompted variant, and §5.4 automatic-mask variant). SAM is invoked through the open-source `segment-anything` Python package on the raw RGB image. The selection of the droplet among SAM's candidate masks (the shape-based filtering rules in §5.4) is our own.

### Generative-AI used during development
- **Anthropic Claude** was used as a coding and writing assistant throughout the project. Specifically:
  - **Code:** Claude assisted in writing the four segmentation methods, the geometric circle-fit for contact angles, the runner scripts, and this notebook. All code was reviewed, tested and adapted by us; we can explain in our own words how every part of the final pipeline works.
  - **Writing:** Claude helped draft the README and parts of this notebook. The content (results, decisions, discussion) is ours; Claude was used to phrase and structure it.
  - **Debugging:** Claude was used to diagnose specific failure modes (the baseline-detector issue in §5.5, the circle-fit bias in §7.2). The diagnoses were verified empirically by us before being acted on.

### Verification
- All results in this notebook were produced by running the code on the actual dataset and saved under `automatic_segmentation/results/`. The notebook re-loads these results for display; the heavy computations (SAM on all 39 images) are reproduced by `automatic_segmentation/reproduce/run_method.py --method automask`.

### What was *not* done
- No part of the pipeline uses generative AI to fabricate, hallucinate, or otherwise infer image content beyond the segmentation step described in §5.
- No contact-angle value in this notebook was generated by a language model; every angle was computed by the geometric circle fit on a real binary mask.

---

*End of notebook. Code modules: `automatic_segmentation/methods/`. Re-running everything from scratch: `automatic_segmentation/reproduce/`. Full project README: `README.md` at the repo root.*